In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=512,
    chunk_overlap=128,
    length_function=len,
    is_separator_regex=False,
)

In [2]:
from openai import OpenAI
import os 
from dotenv import load_dotenv
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
###INSERT CONTEXT
def insertContext(text, full_doc):
    content = text.page_content
    res  = client.chat.completions.create(
    model="gpt-4o",
    store=True,
    messages=[
        {"role": "system", "content": f"please generate appropriate context for the provided chunk. Please note that the added context should include information that is in the following document but not in the chunk. Documnet: \n {full_doc}."},
        {"role": "user", "content": f"chunk: {content}"}
    ])
    context = res.choices[0].message.content
    cached_tokens = res.usage.prompt_tokens_details.cached_tokens
    return content + context 

In [4]:
from transformers import BertTokenizer

# load bert tokenizer from huggingface
tokenizer = BertTokenizer.from_pretrained(
    'bert-base-uncased'
)

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [5]:
from collections import Counter

def build_dict(input_batch):
  # store a batch of sparse embeddings
    sparse_emb = []
    # iterate through input batch
    for token_ids in input_batch:
        # convert the input_ids list to a dictionary of key to frequency values
        d = dict(Counter(token_ids))
        tokenids = list(set(token_ids))
        # remove special tokens and append sparse vectors to sparse_emb list
        # sparse_emb.append({key: d[key] for key in d if key not in [101, 102, 103, 0]})
        sparse_emb.append({"indices":tokenids, "values":[float(d[id]) for id in tokenids]})
    # return sparse_emb list
    return sparse_emb

In [6]:
def generate_sparse_vectors(context_batch):
    input_ids = tokenizer(
    context_batch, padding=True, truncation=True,
     max_length=512
)["input_ids"]
    sparse_embeds = build_dict(input_ids)
    return sparse_embeds

In [7]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
pc = Pinecone(api_key= os.getenv("PINECONE_API_KEY_500"))
index_name = "contextual-retriever"


In [16]:
indices = []
for index in pc.list_indexes():
    indices.append(index["name"])

if index_name in indices :
    print(f"index {index_name} already exists!")
else:
    pc.create_index(
  name=index_name,
  dimension=3072,
  metric="dotproduct",
  spec=ServerlessSpec(
    cloud="aws",
    region="us-east-1"
  ),
  deletion_protection="disabled"
)
    print(f"index {index_name} created")



index contextual-retriever already exists!


In [17]:
index = pc.Index("contextual-retriever")